# Fine-Tuning GPT-4.1 for Video Motion Recognition

This notebook demonstrates vision fine-tuning over frames sampled from short video clips. The default path builds three non-human motion patterns from CIFAR-10 vehicle images, so the complete workflow can run on a standard fine-tuning resource without special content-filter approval.

UCF101 remains available through `VIDEO_DATA_SOURCE=huggingface` or `VIDEO_DATA_SOURCE=kaggle` for resources approved to fine-tune images containing people. UCF101 attribution: https://www.crcv.ucf.edu/research/data-sets/ucf101/ and https://arxiv.org/abs/1212.0402.

__Content:__

1. Retrieve dataset and specify scope
2. Prepare dataset for fine-tuning
3. Configure and start fine-tuning job
4. Deploy fine-tuned model
5. Test inference
6. Base versus fine-tuned model evaluation

## Setup

In [ ]:
import json
import os
from pathlib import Path
from io import StringIO
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
from openai import AzureOpenAI
from sklearn.metrics import classification_report
from tqdm import tqdm
import kagglehub
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentModel, DeploymentProperties, Sku

%config InlineBackend.figure_format = 'retina'

from VideoFTTools import DatasetHelper, VideoExtractor, VideoAnalyzer, Evaluator
from VideoFTTools import upload_frame_to_blob_as_jpeg, date_sorted_df

In [ ]:
# Load .env when present; process environment variables also work.
load_dotenv(find_dotenv())

client_options = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "api_version": os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview"),
}
if os.getenv("AZURE_OPENAI_API_KEY"):
    client_options["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]
else:
    client_options["azure_ad_token_provider"] = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    )
client = AzureOpenAI(**client_options)

project_name = os.getenv("VIDEO_PROJECT_NAME", "action-recognition-ft")
version = os.getenv("VIDEO_VERSION", "smoke")
ft_deployment = os.getenv("VIDEO_FT_DEPLOYMENT", f"{project_name}-{version}")
base_deployment = os.getenv("VIDEO_BASE_DEPLOYMENT", "gpt-4.1")

top_n_classes = int(os.getenv("VIDEO_CLASS_COUNT", "10"))
train_samples = int(os.getenv("VIDEO_TRAIN_SAMPLES", "300"))
val_samples = int(os.getenv("VIDEO_VAL_SAMPLES", "80"))
test_samples = int(os.getenv("VIDEO_TEST_SAMPLES", "80"))
no_frames = int(os.getenv("VIDEO_FRAMES", "5"))
random_seed = 0

container_name = "frames"
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID") or os.getenv("SUBSCRIPTION_ID")
resource_name = os.getenv("AZURE_ACCOUNT_NAME") or os.getenv("AOAI_RESOURCE")
rg_name = os.getenv("AZURE_RESOURCE_GROUP") or os.getenv("RESOURCE_GROUP")

bold_start, bold_end = "\033[1m", "\033[0m"


## Retrieve dataset and specify scope

The default dataset generator creates deterministic horizontal, vertical, and zoom clips from CIFAR-10 vehicle images, mapped to neutral class labels. Neutral labels make the baseline-versus-fine-tuned comparison test learned task adaptation rather than recognition of descriptive class names.

Set `VIDEO_DATA_SOURCE` to `huggingface` or `kaggle` to use UCF101 instead. The Kaggle path requires credentials in `.env`.

In [ ]:
# The default path creates motion clips from non-human CIFAR-10 images.
# UCF101 remains available for resources approved to train on images
# containing people.
dataset_source = os.getenv("VIDEO_DATA_SOURCE", "synthetic").lower()
if dataset_source == "synthetic":
    import pickle
    import tarfile

    data_root = Path(os.getenv("VIDEO_DATA_DIR", "./data"))
    dataset_root = data_root / "cifar_camera_motion"
    cifar_root = data_root / "cifar-10-batches-py"
    if not cifar_root.exists():
        data_root.mkdir(parents=True, exist_ok=True)
        archive_path = data_root / "cifar-10-python.tar.gz"
        if not archive_path.exists():
            with requests.get(
                "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
                stream=True,
                timeout=120,
            ) as response:
                response.raise_for_status()
                with archive_path.open("wb") as target:
                    for chunk in response.iter_content(1024 * 1024):
                        target.write(chunk)
        with tarfile.open(archive_path) as bundle:
            bundle.extractall(data_root, filter="data")

    images, image_labels = [], []
    for batch_name in [f"data_batch_{index}" for index in range(1, 6)] + ["test_batch"]:
        with (cifar_root / batch_name).open("rb") as source:
            batch = pickle.load(source, encoding="bytes")
        batch_images = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        images.extend(batch_images)
        image_labels.extend(batch[b"labels"])
    safe_images = [
        image for image, label in zip(images, image_labels)
        if label in {0, 1, 8, 9}
    ]
    labels = ["motion_alpha", "motion_beta", "motion_gamma"]

    def write_synthetic_split(split, count):
        split_root = dataset_root / split
        expected = split_root / ".complete"
        existing_count = len(list(split_root.glob("*/*.mp4"))) if split_root.exists() else 0
        if expected.exists() and existing_count >= count:
            return
        split_root.mkdir(parents=True, exist_ok=True)
        offsets = {"train": 0, "val": 1, "test": 2}
        rng = np.random.default_rng(random_seed + offsets[split])
        selected = rng.choice(len(safe_images), size=count, replace=False)
        for index in range(count):
            label = labels[index % len(labels)]
            label_root = split_root / label
            label_root.mkdir(parents=True, exist_ok=True)
            path = label_root / f"{label}-{index:03d}.mp4"
            writer = cv2.VideoWriter(
                str(path), cv2.VideoWriter_fourcc(*"mp4v"), 10, (96, 96)
            )
            source_image = cv2.cvtColor(safe_images[selected[index]], cv2.COLOR_RGB2BGR)
            for frame_index in range(24):
                progress = frame_index / 23
                if label == "motion_alpha":
                    canvas = cv2.resize(source_image, (128, 96), interpolation=cv2.INTER_CUBIC)
                    x = int(32 * progress)
                    frame = canvas[:, x:x + 96]
                elif label == "motion_beta":
                    canvas = cv2.resize(source_image, (96, 128), interpolation=cv2.INTER_CUBIC)
                    y = int(32 * progress)
                    frame = canvas[y:y + 96, :]
                else:
                    size = int(96 + 48 * progress)
                    canvas = cv2.resize(source_image, (size, size), interpolation=cv2.INTER_CUBIC)
                    offset = (size - 96) // 2
                    frame = canvas[offset:offset + 96, offset:offset + 96]
                writer.write(frame)
            writer.release()
        expected.write_text("generated", encoding="utf-8")

    write_synthetic_split("train", max(train_samples, 30))
    write_synthetic_split("val", max(val_samples, 12))
    write_synthetic_split("test", max(test_samples, 12))
    dataset_path = str(dataset_root)
elif dataset_source == "kaggle":
    dataset_path = kagglehub.dataset_download(
        "matthewjansen/ucf101-action-recognition"
    )
elif dataset_source == "huggingface":
    import tarfile
    from huggingface_hub import hf_hub_download

    archive = hf_hub_download(
        repo_id="sayakpaul/ucf101-subset",
        filename="UCF101_subset.tar.gz",
        repo_type="dataset",
    )
    extraction_root = Path(os.getenv("VIDEO_DATA_DIR", "./data"))
    dataset_root = extraction_root / "UCF101_subset"
    if not dataset_root.exists():
        extraction_root.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as bundle:
            bundle.extractall(extraction_root, filter="data")
    dataset_path = str(dataset_root)
else:
    raise ValueError("VIDEO_DATA_SOURCE must be 'synthetic', 'huggingface', or 'kaggle'")

print("Path to dataset files:", dataset_path)
print(os.listdir(dataset_path))


In [ ]:
# Load Kaggle metadata or construct equivalent metadata from the compact archive.
if dataset_source == "kaggle":
    orig_train_df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
    orig_val_df = pd.read_csv(os.path.join(dataset_path, "val.csv"))
    orig_test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    for frame in (orig_train_df, orig_val_df, orig_test_df):
        frame["clip_path"] = frame["clip_path"].str.lstrip("/")
else:
    def split_dataframe(split):
        split_root = Path(dataset_path) / split
        rows = [
            {
                "clip_path": str(video.relative_to(dataset_path)),
                "label": video.parent.name,
            }
            for video in split_root.glob("*/*")
            if video.is_file() and video.suffix.lower() in {".avi", ".mp4"}
        ]
        return pd.DataFrame(rows)

    orig_train_df = split_dataframe("train")
    orig_val_df = split_dataframe("val")
    orig_test_df = split_dataframe("test")

dataset_helper = DatasetHelper()
dataset_helper.plot_label_counts(orig_train_df, "Original training dataset")


We are reducing the dataset complexity and size in two ways for quick development iterations:

1. Reduce number of dataset classes
2. Reduce train, val, and test splits to specified number of observations

Feel free to adjust the number of classes and instances for your own experiments.

In [ ]:
top_classes = orig_train_df["label"].value_counts().index[:top_n_classes]

def sample_rows(frame, count):
    selected = frame[frame["label"].isin(top_classes)]
    if count:
        selected = selected.sample(
            n=min(count, len(selected)),
            random_state=random_seed,
        )
    return selected.reset_index(drop=True)

train_df = sample_rows(orig_train_df, train_samples)
val_df = sample_rows(orig_val_df, val_samples)
test_df = sample_rows(orig_test_df, test_samples)

dataset_helper.plot_label_counts(train_df, "Reduced training dataset")
dataset_helper.plot_label_counts(val_df, "Reduced validation dataset")
dataset_helper.plot_label_counts(test_df, "Reduced test dataset")


## Prepare dataset for fine-tuning
Language models like GPT-4.1 are not capable of processing video files directly. Instead, we need to extract individual frames as images to send to the model. For instance, a short 20-second video recorded at 25 fps could contain up to 500 frames, making it computationally expensive to process all of them. To address this, we adopt a frame sampling strategy.

Frame sampling strategies reduce computational complexity and can be tailored to specific use cases. Common strategies include:

- **Sample at specified intervals**: Extract frames at regular intervals (e.g., every 0.25 seconds).
- **Sample a fixed number of frames**: Select a specified number of frames evenly distributed across the video duration.
- **Scene-based sampling**: Identify distinct scenes in the video and sample frames from each scene.
- **Drop similar frames**: Remove redundant frames that contain minimal variation.

These strategies can also be combined to meet more complex requirements.

To begin, we examine the durations of the videos in the training dataset using the `DatasetHelper.plot_video_duration_histogram()` method. This analysis provides insights into the distribution of video lengths, helping to inform our sampling approach.


In [ ]:
dataset_helper.plot_video_duration_histogram(train_df, dataset_path)


The `VideoExtractor.extract_n_video_frames(n)` method extracts `n` frames from a video file, distributing them evenly across the video's duration. Additionally, it appends a timestamp strip to each frame, providing the model with temporal context to understand the sequence and timing of events in the video.

In [ ]:
# pick a random video
video_path = os.path.join(dataset_path, train_df.sample(n=1)['clip_path'].iloc[0])

video_extractor = VideoExtractor(video_path)

frames = video_extractor.extract_n_video_frames(n=no_frames)
video_extractor.display_frames(frames, height=180)

Since the videos in our dataset are quite short and most of them consist of just one scene, we decide to sample only 5 frames per video.

Fine-tuning for video frames is possible with JSONL dataset files similar to the process of sending images as input to the chat completion API.
Images can be provided as HTTP URLs or data URLs containing base64-encoded images.

The next cells are for illustration purposes only (no need to execute).

_Example of video frames provided as **URLs**:_

In [ ]:
{
    "messages": [
        {
            "role": "system",
            "content": "You identify motion patterns in video frames. Choose one label from ['motion_alpha', 'motion_beta', 'motion_gamma'] and respond with a JSON object: {\"activity\": \"label\"}."
        },
        {
            "role": "user",
            "content": "These are the video frames for activity identification."
        },
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": "https://example.com/frames/shavingbeard/frame1.jpg"}},
                {"type": "image_url", "image_url": {"url": "https://example.com/frames/shavingbeard/frame2.jpg"}},
                {"type": "image_url", "image_url": {"url": "https://example.com/frames/shavingbeard/frame3.jpg"}}
            ]
        },
        {
            "role": "assistant",
            "content": "{\"activity\": \"ShavingBeard\"}"
        }
    ]
}

_Example of video frames provided as **base64 encoded images**:_

In [ ]:
{
    "messages": [
        {
            "role": "system",
            "content": "You identify motion patterns in video frames. Choose one label from ['motion_alpha', 'motion_beta', 'motion_gamma'] and respond with a JSON object: {\"activity\": \"label\"}."
        },
        {
            "role": "user",
            "content": "These are the video frames for activity identification."
        },
        {
            "role": "user",
            "content": [
                {"type": "image_base64", "image_data": "data:image/jpg;base64,/9j/4AAQ..."},
                {"type": "image_base64", "image_data": "data:image/jpg;base64,/9j/4AAQ..."},
                {"type": "image_base64", "image_data": "data:image/jpg;base64,/9j/4AAQ..."}
            ]
        },
        {
            "role": "assistant",
            "content": "{\"activity\": \"ShavingBeard\"}"
        }
    ]
}

In [ ]:
top_classes_str = ', '.join(top_classes)

system_message = f"""You are an expert in identifying motion patterns in video clips. 
You are provided with a series of extracted frames from a video. Each frame includes a timestamp in the lower-left corner, formatted as 'video_time: mm:ss:msec'. Use these timestamps to understand the sequence and context of the video.

Your task is to identify the single motion pattern depicted in the video frames. The identified pattern must be one of the following:
{top_classes_str}

Provide the result as a valid JSON object in the exact format below:
{{
    "activity": "The identified activity from the provided list, spelled exactly as given." (string)
}}
"""

The below function generates local JSONL dataset files in the format required for GPT-4.1 vision fine-tuning. The files are uploaded to the Azure OpenAI Fine-tuning API later. Use the `image_source` parameter to specify how to provide the video frames:
- `base64`: Frames will be included in the JSONL file directly as base64-encoded strings. 
- `url`: Frames will be uploaded to an Azure Storage Blob container that you provide and an URL pointing to the video frames will be generated.

In [ ]:
# defining the function here so that it can be adjusted to other dataset formats directly
def generate_ft_dataset(
    df, 
    frames_per_video: int = 5, 
    image_source: str = 'base64',
    dataset_path: str = dataset_path,
    system_message: str = system_message,
    container_name: str = container_name,
    connection_string: str = connection_string
) -> List[Dict]:
    """
    Generates a dataset from video clips by extracting frames and including their metadata.

    Args:
        df: A pandas DataFrame containing video metadata. Must include `clip_path` and `label` columns.
        frames_per_video (int): Number of frames to extract from each video. Default is 5.
        image_source (str): Specifies the source of the image data ('base64' or 'url').
        dataset_path (str): Path to the dataset containing video files. Default is './datasets'.
        system_message (str): A system message to initialize the dataset entry. Default is "Process video frames".
        container_name (str): Azure Blob Storage container name for uploading images. Default is "frames".
        connection_string (str): Azure Blob Storage connection string.

    Returns:
        List[Dict]: A list of dataset entries, each containing the frames and associated labels.

    Raises:
        ValueError: If `image_source` is not 'base64' or 'url'.
        RuntimeError: If frame extraction or Azure upload fails.
    """
    dataset = []

    for index, video in tqdm(df.iterrows(), total=df.shape[0], desc="Processing videos"):
        clip_path = video.get('clip_path')
        label = video.get('label')
        
        if not clip_path or not label:
            print(f"Skipping entry at index {index}: Missing 'clip_path' or 'label'")
            continue

        full_path = os.path.join(dataset_path, clip_path)

        # Extract video frames
        video_extractor = VideoExtractor(uri=full_path)
        try:
            frames = video_extractor.extract_n_video_frames(n=frames_per_video)
        except Exception as e:
            print(f"Failed to extract frames for video {clip_path}: {e}")
            continue

        # Initialize message
        message = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": "These are the frames from the video."}
        ]

        if image_source == 'base64':
            try:
                frames_list = [entry['frame_base64'] for entry in frames]
                message.append({"role": "user", "content": [
                    *map(lambda x: {"type": "image_url",
                                    "image_url": {"url": f'data:image/jpg;base64,{x}'}}, frames_list),
                ]})
            except KeyError:
                print(f"Missing 'frame_base64' key in frames for video {clip_path}")
                continue

        elif image_source == 'url':
            frame_urls = []
            for frame in frames:
                try:
                    video_name = os.path.splitext(os.path.basename(clip_path))[0]
                    timestamp_safe = frame['timestamp'].replace(':', '-')
                    blob_name = f"{clip_path}/{video_name}-{timestamp_safe}.jpg"

                    # Upload frame to Azure Blob Storage
                    blob_url = upload_frame_to_blob_as_jpeg(
                        base64_frame=frame['frame_base64'],
                        connection_string=connection_string,
                        container_name=container_name,
                        blob_name=blob_name
                    )
                    frame_urls.append({"type": "image_url", "image_url": {"url": blob_url}})
                except Exception as e:
                    print(f"Failed to upload frame for video {clip_path}: {e}")
                    continue

            message.append({"role": "user", "content": frame_urls})

        else:
            raise ValueError(f"Invalid image_source: {image_source}. Must be 'base64' or 'url'.")

        # Add assistant response with ground truth label
        message.append({"role": "assistant", "content": json.dumps({"activity": label})})

        dataset.append({"messages": message})

    return dataset

In [ ]:
# generate training file
output_file_path = f"{project_name}-{version}-train.jsonl"
print(f"Creating dataset: {output_file_path}")

train_ds = generate_ft_dataset(train_df, image_source='base64')

with open(output_file_path, "w") as jsonl_file:
    for message in train_ds:
        jsonl_file.write(json.dumps(message) + "\n")

print(f"File size: {os.path.getsize(output_file_path) / (1024 * 1024):.2f} MB")

In [ ]:
# generate validation file
output_file_path = f"{project_name}-{version}-val.jsonl"
print(f"Creating dataset: {output_file_path}")

val_ds = generate_ft_dataset(val_df, image_source='base64')

with open(output_file_path, "w") as jsonl_file:
    for message in val_ds:
        jsonl_file.write(json.dumps(message) + "\n")

print(f"File size: {os.path.getsize(output_file_path) / (1024 * 1024):.2f} MB")

## Configure and start fine-tuning job

In [ ]:
# Upload and wait for processing before starting fine-tuning.
import time

def upload_processed(path):
    with open(path, "rb") as source:
        uploaded = client.files.create(file=source, purpose="fine-tune")
    while True:
        for attempt in range(5):
            try:
                uploaded = client.files.retrieve(uploaded.id)
                break
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        if uploaded.status == "processed":
            return uploaded
        if uploaded.status == "error":
            raise RuntimeError(
                f"File {uploaded.id} failed processing: {uploaded.status_details}"
            )
        time.sleep(2)

resume_job_id = os.getenv("VIDEO_REUSE_FT_JOB_ID")
if not resume_job_id:
    train_file = upload_processed(f"{project_name}-{version}-train.jsonl")
    val_file = upload_processed(f"{project_name}-{version}-val.jsonl")


Here is some guidance if you want to adjust the hyperparameters of the fine-tuning process. You can keep them as `None` to use default values. 

| Hyperparameter                       | Description                                                                                                                                                                              |
|-----------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Batch size`                            | The batch size to use for training. When set to default, batch_size is calculated as 0.2% of examples in training set and the max is 256.                                                           |
| `Learning rate multiplier` | The fine-tuning learning rate is the original learning rate used for pre-training multiplied by this multiplier. We recommend experimenting with values between 0.5 and 2. Empirically, we've found that larger learning rates often perform better with larger batch sizes. Must be between 0.0 and 5.0. |
| `Number of epochs`       | Number of training epochs. An epoch refers to one full cycle through the data set. If set to default, number of epochs will be determined dynamically based on the input data. |
| `Seed`  | The seed controls the reproducibility of the job. Passing in the same seed and job parameters should produce the same results, but may differ in rare cases. If a seed is not specified, one will be generated for you. |

In [ ]:
# Create a fine-tuning job, or resume the job from an interrupted notebook run.
if resume_job_id:
    ft_job = client.fine_tuning.jobs.retrieve(resume_job_id)
    print(f"Resuming fine-tuning job {ft_job.id}")
else:
    ft_job = client.fine_tuning.jobs.create(
        suffix=f"{project_name}-{version}",
        training_file=train_file.id,
        validation_file=val_file.id,
        model=os.getenv("VIDEO_FT_MODEL", "gpt-4.1-2025-04-14"),
        method={
            "type": "supervised",
            "supervised": {
                "hyperparameters": {
                    "n_epochs": int(os.getenv("VIDEO_FT_EPOCHS", "1"))
                }
            },
        },
    )
    print(f"Submitted fine-tuning job {ft_job.id}")


In [ ]:
# Wait for fine-tuning to reach a terminal state.
while ft_job.status not in {"succeeded", "failed", "cancelled"}:
    time.sleep(30)
    for attempt in range(5):
        try:
            ft_job = client.fine_tuning.jobs.retrieve(ft_job.id)
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)
    print(f"Fine-tuning status: {ft_job.status}")

if ft_job.status != "succeeded":
    events = client.fine_tuning.jobs.list_events(
        fine_tuning_job_id=ft_job.id,
        limit=20,
    )
    event_messages = "\n".join(event.message for event in events.data)
    raise RuntimeError(
        f"Fine-tuning ended with status {ft_job.status}: {ft_job.error}\n"
        f"{event_messages}"
    )
display(ft_job.to_dict())


In [ ]:
# In case you want to retrieve a specific job 
# ft_job = client.fine_tuning.jobs.retrieve("<fine-tuning-job-id>")

In [ ]:
display(ft_job.to_dict())

In [ ]:
# Retrieve fine-tuning metrics from result file

result_file_id = ft_job.to_dict()['result_files'][0]
results_content = client.files.content(result_file_id).content.decode()

data_io = StringIO(results_content)

results_df = pd.read_csv(data_io)

evaluator = Evaluator()
evaluator.plot_learning_curves(results_df, smoothing_window=5)

Take a look at this table for an interpretation of above diagrams:  

| Metric                       | Description                                                                                                                                                                              |
|-----------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `step`                            | The number of the training step. A training step represents a single pass, forward and backward, on a batch of training data.                                                           |
| `train_loss`, `validation_loss` | The loss for the training / validation batch |
| `train_mean_token_accuracy`       | This metric represents the average accuracy of the model in predicting the correct token during the training phase. It is calculated by comparing the model's predicted tokens to the actual tokens in the training dataset. A higher value indicates better performance on the training data. For example, if the batch size is set to 3 and your data contains completions [[1, 2], [0, 5], [4, 2]], this value is set to 0.83 (5 of 6) if the model predicted [[1, 1], [0, 5], [4, 2]]. |
| `validation_mean_token_accuracy`  | This metric measures the average accuracy of the model in predicting the correct token on a separate validation dataset. It serves as an indicator of the model's ability to generalize to new, unseen data. Consistently low validation accuracy, especially when training accuracy is high, may suggest issues such as overfitting. |

## Deploy fine-tuned model
You can use the Azure AI Foundry portal to perform a manual deployment from your job under the __Fine-tuning__ menu.

If you prefer an automated  approach, the following code shows how to deploy the model using the Control Plane API. Take a look at the [Azure OpenAI fine-tuning documentation](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/fine-tuning?tabs=turbo&pivots=programming-language-python#deploy-fine-tuned-model) for more details.

In [ ]:
# List existing models
my_models = client.models.list().to_dict()

models_df = date_sorted_df(my_models['data'])

cols = ['status', 'capabilities', 'lifecycle_status', 'id', 'created_at', 'model']
print(f'Models of AOAI resource {bold_start}{resource_name}{bold_end}:')
display(models_df[cols].head())

In [ ]:
fine_tuned_model = ft_job.fine_tuned_model

credential = DefaultAzureCredential()
deployment_config = Deployment(
    properties=DeploymentProperties(
        model=DeploymentModel(
            format="OpenAI",
            name=fine_tuned_model,
            version="1",
        )
    ),
    sku=Sku(
        name=os.getenv("VIDEO_FT_SKU", "GlobalStandard"),
        capacity=int(os.getenv("VIDEO_FT_CAPACITY", "50")),
    ),
)
with CognitiveServicesManagementClient(credential, subscription_id) as management_client:
    management_client.deployments.begin_create_or_update(
        rg_name,
        resource_name,
        ft_deployment,
        deployment_config,
    ).result()

print(f"Fine-tuned deployment ready: {ft_deployment}")


## Test Inference

In [ ]:
video_analyzer = VideoAnalyzer(client, ft_deployment)
display(test_df.head())

In [ ]:
# pick a random video

sample = test_df.sample(n=1).iloc[0]
ground_truth_label = sample['label']
clip_path = sample['clip_path']

video_path = os.path.join(dataset_path, clip_path)

video_extractor = VideoExtractor(video_path)

frames = video_extractor.extract_n_video_frames(n=no_frames)
video_extractor.display_frames(frames, height=180)

In [ ]:
frames_list = [entry['frame_base64'] for entry in frames]

llm_insights = video_analyzer.video_chat(frames_list, transcription=None, system_message=system_message)

print(f"Predicted activity: {llm_insights['activity']}")
print(f"Actual activity: {ground_truth_label}")

## Base versus Fine-Tuned Model Evaluation

The following code can be used to systematically evaluate the base model (GPT-4.1 in this case) against the fine-tuned model on the test dataset. This comparison helps quantify the improvements achieved through fine-tuning.

In [ ]:
def classify_video(clip_path, video_analyzer, local_path=None, frames_per_video=1):
    """ Classify a single video and return its predicted activity. """

    uri = os.path.join(local_path, clip_path) if local_path else clip_path
    video_extractor = VideoExtractor(uri)
    frames = video_extractor.extract_n_video_frames(n=frames_per_video)
    frames_list = [entry['frame_base64'] for entry in frames]
    llm_insights = video_analyzer.video_chat(frames_list, transcription=None, system_message=system_message)
    return llm_insights['activity']


def classify_videos_in_df(test_df, video_analyzer, local_path, frames_per_video, column_name):
    """ Classify videos in a DataFrame and add predictions to a specified column. """
    tqdm.pandas(desc=f"Processing {column_name}")
    test_df[column_name] = test_df.progress_apply(
        lambda row: classify_video(
            clip_path=row['clip_path'],
            video_analyzer=video_analyzer,
            local_path=local_path,
            frames_per_video=frames_per_video
        ), axis=1
    )
    return test_df

In [ ]:
# Base and fine-tuned model evaluation
base_video_analyzer = VideoAnalyzer(client, base_deployment)
ft_video_analyzer = VideoAnalyzer(client, ft_deployment)

# Classify with the base model
test_df = classify_videos_in_df(
    test_df=test_df,
    video_analyzer=base_video_analyzer,
    local_path=dataset_path,
    frames_per_video=no_frames,
    column_name='base_predicted_label'
)

# Classify with the fine-tuned model
test_df = classify_videos_in_df(
    test_df=test_df,
    video_analyzer=ft_video_analyzer,
    local_path=dataset_path,
    frames_per_video=no_frames,
    column_name='ft_predicted_label'
)

In [ ]:
# save evaluation results
test_df.to_csv(f"{project_name}-{version}-eval-results-{test_df.shape[0]}-entries.csv", index=False)

In [ ]:
# Optionally reload a previously saved evaluation.
previous_results = os.getenv("VIDEO_PREVIOUS_RESULTS")
if previous_results:
    test_df = pd.read_csv(previous_results)
    display(test_df.head())


In [ ]:
evaluator.compare_model_metrics(test_df)

After comparing the performance of the base vs. fine-tuned model, we can further analyze the fine-tuning results by inspecting the **classification report** and **confusion matrix**.

In [ ]:
y_true = test_df['label']
y_pred = test_df['ft_predicted_label']

# Generate classification report
report = classification_report(y_true, y_pred, output_dict=False, zero_division=True)
print(report)

In [ ]:
evaluator = Evaluator()

evaluator.plot_confusion_matrix(y_true, y_pred)